# Module 22 — Streamlit

The same numbers as module 21, and the sharpest comparison in Part 5: **no routes, no
templates, no HTML, and no `url_for`.** `app.py` next to this notebook is under a
hundred lines and there is not a tag in it.

What replaces all of that is an execution model, and it is the one thing in this
module you have to actually understand: **the whole script runs again, top to bottom,
on every interaction.** Everything else follows from that — why widgets return values,
why a plain variable is useless, why `@st.cache_data` exists, and where Streamlit stops
being the right answer.

Testing works without a browser: `AppTest` runs the script and hands you the elements
it produced, so every cell below is checkable.

In [ ]:
from pathlib import Path

import streamlit as st
from streamlit.testing.v1 import AppTest

from sensorreport import LIMIT, load_readings

HERE = Path.cwd() if (Path.cwd() / "app.py").is_file() else Path.cwd() / "22_streamlit"

print(st.__version__, "| readings:", len(load_readings()), "| limit:", LIMIT)

## 1. The script *is* the page

There is no route and no template. Statements run in order, and each one that starts
with `st.` puts something on the page **at that position**.

In [ ]:
SCRIPT = """
import streamlit as st
from sensorreport import load_readings, summarise

st.title("Sensor summary")
st.write("One row per location.")

for s in summarise(load_readings()):
    st.write(f"{s.location}: {s.mean}")
"""

app = AppTest.from_string(SCRIPT, default_timeout=30)
app.run()

print("exceptions:", [e.value for e in app.exception] or "none")
print("title:", app.title[0].value)
print("text:", [item.value for item in app.markdown])

`AppTest.from_string` runs the script in-process and collects what it produced, by
type: `app.title`, `app.markdown`, `app.metric`, `app.slider`. That is module 15's
test client for a Streamlit app, and it needs no browser and no port.

`default_timeout=30` is not decoration. The default is **three seconds**, and a
Streamlit script that reads a file and draws a chart can exceed that on a loaded
machine — which produces a test that passes alone and fails in a full suite. A test
that depends on the wall clock is a flaky test, and a flaky test is worse than no
test: it teaches people to re-run until it goes green.

The order of the output is the order of the statements. Which sounds obvious and is
the whole design: you are not describing a page, you are printing one.

## 2. A widget returns its value

`st.slider(...)` does two things: it puts a slider on the page, and it **returns the
value it currently has.** On the first run that is the default.

In [ ]:
SCRIPT = """
import streamlit as st
from sensorreport import load_readings

above = st.slider("above", 0.0, 100.0, 85.0)
hot = [r for r in load_readings() if r.value is not None and r.value > above]
st.metric("faults", len(hot))
"""

app = AppTest.from_string(SCRIPT, default_timeout=30)
app.run()

print("slider:", app.slider[0].value, "-> metric:", app.metric[0].value)

Now move it. Predict what the metric says afterwards — there are 50 readings, 3 of
them unreadable.

In [ ]:
SCRIPT = """
import streamlit as st
from sensorreport import load_readings

above = st.slider("above", 0.0, 100.0, 85.0)
hot = [r for r in load_readings() if r.value is not None and r.value > above]
st.metric("faults", len(hot))
"""

app = AppTest.from_string(SCRIPT, default_timeout=30)
app.run()
app.slider[0].set_value(20.0).run()

# The metric is a string. How many readings are above 20?
assert app.metric[0].value == ...

Which means something has to have happened between the two runs, and here it is:
**the whole script ran again.** The slider was not a callback and there was no event
handler. The file was executed a second time, from the first line, and this time
`st.slider(...)` returned `20.0` instead of `85.0`.

That is the model in one sentence. Everything else in this module is a consequence.

## 3. The first consequence: variables do not survive

If the file runs again from the top, then every local variable is created again from
the top. A counter in an ordinary variable counts to one, for ever.

In [ ]:
SCRIPT = """
import streamlit as st

runs = 0
runs += 1
st.write(f"plain={runs}")

st.session_state.setdefault("kept", 0)
st.session_state["kept"] += 1
st.write(f"kept={st.session_state['kept']}")

st.button("again")
"""

app = AppTest.from_string(SCRIPT, default_timeout=30)
app.run()
app.button[0].click().run()
app.button[0].click().run()

# Three runs. What do the two lines say?
assert [item.value for item in app.markdown] == ...

`plain=1` and `kept=3`.

`st.session_state` is a dict that belongs to the **browser session** rather than to the
script run, and it is the only thing that persists. So the rule is:

- **anything derived from the widgets** — a filtered list, a sum, a chart — is computed
  fresh on every run, and that is correct;
- **anything the user built up over several interactions** — a shopping basket, a
  history, a login — goes in `session_state`, or it is gone.

Which also explains why Streamlit code has no `if request.method == "POST"` in it.
There is no request cycle to be in the middle of. There is one script, run again.

## 4. The second consequence: everything expensive runs again

The script re-runs on every keystroke in a text box. If it reads a file at the top,
it reads it on every keystroke. `@st.cache_data` — a decorator, module 14 — is the
answer.

In [ ]:
SCRIPT = """
import streamlit as st

CALLS = []


@st.cache_data
def expensive():
    CALLS.append(1)
    return [1, 2, 3]


expensive()
expensive()
expensive()
st.write(f"calls={len(CALLS)}")
"""

app = AppTest.from_string(SCRIPT, default_timeout=30)
app.run()

print(app.markdown[0].value, "-- three calls, one execution")

`@st.cache_data` keys on the function's arguments and returns a **copy** of the cached
value, so a caller that mutates the result does not corrupt the cache. Its sibling
`@st.cache_resource` returns the same object every time and is for things you do not
want copied — a database connection, a loaded model.

Which is which matters: caching a connection with `cache_data` copies it, and caching
a DataFrame with `cache_resource` hands every session the same object to mutate.

The trap, and it is module 14's trap: **a cached function must be pure.** If it reads a
file, the cache never notices the file changing. Streamlit's own answer is a `ttl=`
argument, and there is a "Clear cache" item in the menu — both of which are admissions
that the function was not pure after all.

## 5. `key=`, and reading a widget from anywhere

A widget's value is normally the return value. Give it a `key` and it is also in
`session_state` under that name, which is how a callback or a later part of the script
reads it.

In [ ]:
SCRIPT = """
import streamlit as st

value = st.slider("limit", 0.0, 100.0, 85.0, key="limit")
st.write(f"returned={value} state={st.session_state['limit']}")
"""

app = AppTest.from_string(SCRIPT, default_timeout=30)
app.run()
print(app.markdown[0].value)

app.slider[0].set_value(20.0).run()
print(app.markdown[0].value)

The two are the same value, and `key` matters for two reasons: it survives into the
next run under a name, and **it identifies the widget**. Streamlit matches widgets
between runs by position and parameters, so a widget inside an `if` that appears and
disappears can lose its value — a `key` is what pins it.

## 6. What an exception does

The script is the page, so an exception ends the page where it happened. Everything
above it is already displayed; everything below never runs.

In [ ]:
SCRIPT = """
import streamlit as st

st.write("this line is displayed")
raise ValueError("something went wrong")
st.write("this line is not")
"""

app = AppTest.from_string(SCRIPT, default_timeout=30)
app.run()

print("shown:", [item.value for item in app.markdown])
print("exception:", [e.value for e in app.exception])

In the browser the traceback appears **in the page**, which is convenient while you
are writing and is the same exposure module 20 described for Flask's debugger: it
shows your code to whoever can reach the port. `client.showErrorDetails` is the
setting that turns it off, and Streamlit's default is to show it.

## 7. The whole application

`app.py` is the same data as module 21, with a slider, three metrics, a table, a chart
and a list. Ninety lines, no HTML.

In [ ]:
app = AppTest.from_file(str(HERE / "app.py"), default_timeout=30)
app.run()

print("exceptions:", [e.value for e in app.exception] or "none")
print("title:   ", app.title[0].value)
print("metrics: ", [(m.label, m.value) for m in app.metric])
print("table:   ", len(app.dataframe[0].value), "rows")
print("slider:  ", app.slider[0].value)

In [ ]:
app = AppTest.from_file(str(HERE / "app.py"), default_timeout=30)
app.run()
app.slider[0].set_value(20.0).run()

print([(m.label, m.value) for m in app.metric])
print("the readings and unreadable counts did not move; the third one did")

## 8. Flask or Streamlit

Both modules show the same numbers. The difference is not size — it is what you are
allowed to control.

| | Flask (21) | Streamlit (22) |
| --- | --- | --- |
| the page | your HTML, in templates | Streamlit's widgets, in its layout |
| URLs | you design them | one page; multipage is a directory convention |
| interaction | a request per click, state in a session or a database | the script re-runs, state in `session_state` |
| a chart | pick a library, embed it | `st.line_chart(data)` |
| who it is for | anybody with a browser and a URL | usually a team, behind a login or a VPN |
| CSS, JavaScript, a design | yours to write | not really available |
| lines for this application | ~70 plus 4 templates | ~90, no templates |

**What Streamlit gives up:** control of the markup, the URL structure, and the request
cycle. You cannot make it look like your company's website, you cannot have
`/location/Hall` as a link somebody bookmarks, and you cannot do anything that needs
to happen without re-running the script.

**What it buys:** a chart is one line, a filter is one line, and there is no HTML
between you and the data. For an analysis somebody needs to *look at and adjust*, that
is the entire job — and it is the same argument module 18 made for a notebook, one step
further along: this is the notebook you can hand to a colleague who will not open
Jupyter.

The heuristic: **is the point the data, or the page?** If a colleague needs to explore
your numbers, Streamlit, and it will take an afternoon. If it is a product with users
who do not work with you, Flask or Django — because sooner or later somebody will ask
for a URL, a login, or a logo in the corner, and those are the three things Streamlit
does not do.

---

`exercises/` is next: seven files to fill in and two to think through. All of them use
`AppTest`, so none of them needs a browser.

Module 23 is FastAPI, where the type hints you have been writing since module 04 stop
being notation and become the interface.